In [1]:
import spacy
import pandas as pd


In [2]:
df_jobs = pd.read_csv('vdab_keywords.csv')

In [3]:
df_jobs.head()

,Unnamed: 0,url,title,employer,location,description,description_translated,title_translated,keywords
0,0,https://www.vdab.be/vindeenjob/vacatures/68793122,technisch aankoopassistent,le grand & associates,kortrijk,Functieomschrijving LGA Engineering is een rek...,Job description LGA Engineering is a recruitme...,technical purchasing assistant,"{'Profile Profile Experience', 'recruitment', ..."
1,1,https://www.vdab.be/vindeenjob/vacatures/68793123,pricing analist,lga engineering,provincie vlaams-brabant,Functieomschrijving Heb jij een passie voor de...,Job description Do you have a passion for the ...,pricing analist,"{'Job', 'career', 'job', 'competencies', 'logi..."
2,2,https://www.vdab.be/vindeenjob/vacatures/68793124,pricing analist maritiem,lga engineering,antwerpen,Functieomschrijving Voor onze logistieke klant...,Job description For our logistics customer in ...,maritime pricing analyst,"{'Job', 'Offer Offer', 'contract prices', 'Ana..."
3,3,https://www.vdab.be/vindeenjob/vacatures/68793125,warehouse teamleader,start people,herentals,Functieomschrijving Onze klant gevestigd in He...,Job description Our client based in Herentals ...,warehouse team leader,"{'operating logistics', 'role', 'globally oper..."
4,4,https://www.vdab.be/vindeenjob/vacatures/68793126,grafisch vormgever,start people,pelt,Functieomschrijving 1 op de 10 gezinnen geniet...,Job description 1 in 10 families already enjoy...,graphical designer,"{'Job', 'designing', 'wooden windows', 'doors'..."


In [4]:
df_jobs[['description_translated','keywords']]

,description_translated,keywords
0,Job description LGA Engineering is a recruitme...,"{'Profile Profile Experience', 'recruitment', ..."
1,Job description Do you have a passion for the ...,"{'Job', 'career', 'job', 'competencies', 'logi..."
2,Job description For our logistics customer in ...,"{'Job', 'Offer Offer', 'contract prices', 'Ana..."
3,Job description Our client based in Herentals ...,"{'operating logistics', 'role', 'globally oper..."
4,Job description 1 in 10 families already enjoy...,"{'Job', 'designing', 'wooden windows', 'doors'..."
...,...,...
11122,Job description We are looking for an installe...,"{'glass', 'West Flanders region', 'GLASS INTER..."
11123,Job description As a forklift driver you are r...,"{'equipment', '3rd grade', 'deviations Report ..."
11124,Job description Expect the unexpected for your...,"{'Technical', 'maintenance', 'supervise', 'res..."
11125,"Job description As a mechanic, your duties inc...","{'equipment', 'mechanical pieces', 'maintenanc..."


In [5]:
# Load the spaCy model
nlp = spacy.load("en_core_web_md")

In [6]:
# Provided categories
categories = {
    "Administration": "Administration involves the performance of executive duties and management functions including planning, organizing, and coordinating office activities and operations to ensure efficiency and adherence to policies. This category includes roles like office managers, administrative assistants, and clerical staff.",
    "Agriculture & fishing": "Agriculture & fishing covers occupations related to farming, crop production, livestock management, aquaculture, and commercial fishing. Roles include farmers, agricultural technicians, fishermen, and related support positions.",
    "Construction & real estate": "Construction & real estate encompasses jobs involved in the planning, design, construction, and sale or leasing of buildings and properties. This includes architects, construction workers, real estate agents, property managers, and surveyors.",
    "Creative jobs": "Creative jobs refer to occupations that involve artistic expression and innovation. This includes roles in the fields of art, design, media, entertainment, and writing such as graphic designers, writers, musicians, and filmmakers.",
    "Education & training": "Education & training covers jobs that focus on teaching, instructing, and developing educational programs. Roles include teachers, trainers, educational administrators, and academic advisors.",
    "Executive board & management": "Executive board & management includes top-level positions responsible for the overall direction and administration of organizations. This includes CEOs, board members, general managers, and other senior executives.",
    "Finance, Insurance & law": "Finance, insurance & law involves jobs related to financial services, insurance policies, legal advice, and representation. Roles include accountants, financial analysts, insurance agents, lawyers, and paralegals.",
    "Government": "Government jobs are positions within public sector organizations that provide services and governance at local, regional, and national levels. This includes civil servants, policy advisors, and public administrators.",
    "Health & body care": "Health & body care includes occupations in healthcare, wellness, and fitness services. Roles include doctors, nurses, therapists, fitness trainers, and beauty consultants.",
    "Hotel, catering, tourism, maintenance": "Hotel, catering, tourism, maintenance covers jobs in the hospitality industry, including lodging, food services, travel, and facility upkeep. Roles include hotel managers, chefs, tour guides, and maintenance workers.",
    "Human resources": "Human resources encompasses roles focused on recruiting, training, employee relations, and organizational development. This includes HR managers, recruiters, and training specialists.",
    "ICT": "ICT (Information and Communications Technology) includes jobs related to computer technology, telecommunications, and information systems. Roles include software developers, IT support specialists, network administrators, and data analysts.",
    "Logistics, transport & purchase": "Logistics, transport & purchase involves the planning and management of supply chains, transportation, and procurement of goods and services. This includes logistics managers, truck drivers, and purchasing agents.",
    "Marketing & communication": "Marketing & communication covers jobs that involve promoting products, services, or brands, and managing public relations. Roles include marketing managers, public relations specialists, and advertising executives.",
    "Research & development": "Research & development includes roles that focus on innovation, scientific research, and product development. This includes scientists, researchers, and R&D managers.",
    "Sales": "Sales involves roles related to selling products or services, managing client relationships, and achieving sales targets. This includes sales representatives, account managers, and retail salespersons.",
    "Social, cultural & sports": "Social, cultural & sports covers jobs in community service, cultural programs, and athletic activities. Roles include social workers, cultural program coordinators, and sports coaches.",
    "Surveillance & security": "Surveillance & security involves jobs focused on protecting people, property, and information. This includes security guards, surveillance operators, and cybersecurity specialists.",
    "Technics, engineering & production": "Technics, engineering & production encompasses jobs in technical fields, engineering disciplines, and manufacturing processes. Roles include engineers, technicians, and production managers.",
    "Trades": "Trades include skilled manual work in areas like carpentry, plumbing, electrical work, and other vocational trades. This includes carpenters, electricians, plumbers, and mechanics."
}

# Function to find the top 3 matching sectors for each keyword
def find_top_matching_sectors(keyword, categories, nlp, top_n=3):
    keyword_vector = nlp(keyword)
    similarities = []
    
    for sector, description in categories.items():
        description_vector = nlp(description)
        similarity = keyword_vector.similarity(description_vector)
        similarities.append((sector, similarity))
    
    # Sort the similarities in descending order and get the top N
    similarities.sort(key=lambda x: x[1], reverse=True)
    top_sectors = [sector for sector, similarity in similarities[:top_n]]
    
    return top_sectors

# Apply the function to the 'job_description' column
df_jobs['top_categories'] = df_jobs['description_translated'].apply(lambda x: find_top_matching_sectors(x, categories, nlp))

df_jobs


,Unnamed: 0,url,title,employer,location,description,description_translated,title_translated,keywords,top_categories
0,0,https://www.vdab.be/vindeenjob/vacatures/68793122,technisch aankoopassistent,le grand & associates,kortrijk,Functieomschrijving LGA Engineering is een rek...,Job description LGA Engineering is a recruitme...,technical purchasing assistant,"{'Profile Profile Experience', 'recruitment', ...","[Administration, Sales, Logistics, transport &..."
1,1,https://www.vdab.be/vindeenjob/vacatures/68793123,pricing analist,lga engineering,provincie vlaams-brabant,Functieomschrijving Heb jij een passie voor de...,Job description Do you have a passion for the ...,pricing analist,"{'Job', 'career', 'job', 'competencies', 'logi...","[Sales, Administration, Creative jobs]"
2,2,https://www.vdab.be/vindeenjob/vacatures/68793124,pricing analist maritiem,lga engineering,antwerpen,Functieomschrijving Voor onze logistieke klant...,Job description For our logistics customer in ...,maritime pricing analyst,"{'Job', 'Offer Offer', 'contract prices', 'Ana...","[Administration, Sales, Logistics, transport &..."
3,3,https://www.vdab.be/vindeenjob/vacatures/68793125,warehouse teamleader,start people,herentals,Functieomschrijving Onze klant gevestigd in He...,Job description Our client based in Herentals ...,warehouse team leader,"{'operating logistics', 'role', 'globally oper...","[Administration, Logistics, transport & purcha..."
4,4,https://www.vdab.be/vindeenjob/vacatures/68793126,grafisch vormgever,start people,pelt,Functieomschrijving 1 op de 10 gezinnen geniet...,Job description 1 in 10 families already enjoy...,graphical designer,"{'Job', 'designing', 'wooden windows', 'doors'...","[Administration, Sales, Creative jobs]"
...,...,...,...,...,...,...,...,...,...,...
11122,11122,https://www.vdab.be/vindeenjob/vacatures/68804504,plaatser - glazen binnenwanden,forum construct maldegem,oostkamp,Functieomschrijving Voor onze klant zijn wij o...,Job description We are looking for an installe...,installer - glass interior walls,"{'glass', 'West Flanders region', 'GLASS INTER...","[Administration, Logistics, transport & purcha..."
11123,11123,https://www.vdab.be/vindeenjob/vacatures/68804505,heftruckchauffeur,forum jobs-food-construct roeselare,roeselare,Functieomschrijving Als heftruckchauffeur sta ...,Job description As a forklift driver you are r...,forklift driver,"{'equipment', '3rd grade', 'deviations Report ...","[Administration, Logistics, transport & purcha..."
11124,11124,https://www.vdab.be/vindeenjob/vacatures/68804506,technisch coördinator (+ bedrijfswagen),square city,evergem,Functieomschrijving Verwacht het onverwachte v...,Job description Expect the unexpected for your...,technical coordinator (+ company car),"{'Technical', 'maintenance', 'supervise', 'res...","[Administration, Sales, Logistics, transport &..."
11125,11125,https://www.vdab.be/vindeenjob/vacatures/68804507,mechanieker,forum jobs-food-construct roeselare,roeselare,Functieomschrijving Als mechanieker behoort to...,"Job description As a mechanic, your duties inc...",mechanic,"{'equipment', 'mechanical pieces', 'maintenanc...","[Logistics, transport & purchase, Administrati..."


In [7]:
df_jobs.to_csv('vdab_categories.csv')

In [13]:
import pandas as pd

In [20]:
df_jobs = pd.read_csv('hr_tool_irakli/VDAB_data_translated_provinces_category.csv')
df_jobs2 = pd.read_csv('vdab_keywords.csv')

In [21]:
df_jobs

,url,title,employer,location,description,description_translated,title_translated,province,category
0,https://www.vdab.be/vindeenjob/vacatures/68793122,technisch aankoopassistent,le grand & associates,kortrijk,Functieomschrijving LGA Engineering is een rek...,Job description LGA Engineering is a recruitme...,technical purchasing assistant,West Flanders,Administration
1,https://www.vdab.be/vindeenjob/vacatures/68793123,pricing analist,lga engineering,provincie vlaams-brabant,Functieomschrijving Heb jij een passie voor de...,Job description Do you have a passion for the ...,pricing analist,Flemish Brabant,"Finance, Insurance & law"
2,https://www.vdab.be/vindeenjob/vacatures/68793124,pricing analist maritiem,lga engineering,antwerpen,Functieomschrijving Voor onze logistieke klant...,Job description For our logistics customer in ...,maritime pricing analyst,Antwerp,Education & training
3,https://www.vdab.be/vindeenjob/vacatures/68793125,warehouse teamleader,start people,herentals,Functieomschrijving Onze klant gevestigd in He...,Job description Our client based in Herentals ...,warehouse team leader,Antwerp,Education & training
4,https://www.vdab.be/vindeenjob/vacatures/68793126,grafisch vormgever,start people,pelt,Functieomschrijving 1 op de 10 gezinnen geniet...,Job description 1 in 10 families already enjoy...,graphical designer,Limburg,Administration
...,...,...,...,...,...,...,...,...,...
11122,https://www.vdab.be/vindeenjob/vacatures/68804504,plaatser - glazen binnenwanden,forum construct maldegem,oostkamp,Functieomschrijving Voor onze klant zijn wij o...,Job description We are looking for an installe...,installer - glass interior walls,West Flanders,Construction & real estate
11123,https://www.vdab.be/vindeenjob/vacatures/68804505,heftruckchauffeur,forum jobs-food-construct roeselare,roeselare,Functieomschrijving Als heftruckchauffeur sta ...,Job description As a forklift driver you are r...,forklift driver,West Flanders,Education & training
11124,https://www.vdab.be/vindeenjob/vacatures/68804506,technisch coördinator (+ bedrijfswagen),square city,evergem,Functieomschrijving Verwacht het onverwachte v...,Job description Expect the unexpected for your...,technical coordinator (+ company car),East Flanders,Administration
11125,https://www.vdab.be/vindeenjob/vacatures/68804507,mechanieker,forum jobs-food-construct roeselare,roeselare,Functieomschrijving Als mechanieker behoort to...,"Job description As a mechanic, your duties inc...",mechanic,West Flanders,Executive board & management


In [22]:
df_jobs2

,Unnamed: 0,url,title,employer,location,description,description_translated,title_translated,keywords
0,0,https://www.vdab.be/vindeenjob/vacatures/68793122,technisch aankoopassistent,le grand & associates,kortrijk,Functieomschrijving LGA Engineering is een rek...,Job description LGA Engineering is a recruitme...,technical purchasing assistant,"{'Profile Profile Experience', 'recruitment', ..."
1,1,https://www.vdab.be/vindeenjob/vacatures/68793123,pricing analist,lga engineering,provincie vlaams-brabant,Functieomschrijving Heb jij een passie voor de...,Job description Do you have a passion for the ...,pricing analist,"{'Job', 'career', 'job', 'competencies', 'logi..."
2,2,https://www.vdab.be/vindeenjob/vacatures/68793124,pricing analist maritiem,lga engineering,antwerpen,Functieomschrijving Voor onze logistieke klant...,Job description For our logistics customer in ...,maritime pricing analyst,"{'Job', 'Offer Offer', 'contract prices', 'Ana..."
3,3,https://www.vdab.be/vindeenjob/vacatures/68793125,warehouse teamleader,start people,herentals,Functieomschrijving Onze klant gevestigd in He...,Job description Our client based in Herentals ...,warehouse team leader,"{'operating logistics', 'role', 'globally oper..."
4,4,https://www.vdab.be/vindeenjob/vacatures/68793126,grafisch vormgever,start people,pelt,Functieomschrijving 1 op de 10 gezinnen geniet...,Job description 1 in 10 families already enjoy...,graphical designer,"{'Job', 'designing', 'wooden windows', 'doors'..."
...,...,...,...,...,...,...,...,...,...
11122,11122,https://www.vdab.be/vindeenjob/vacatures/68804504,plaatser - glazen binnenwanden,forum construct maldegem,oostkamp,Functieomschrijving Voor onze klant zijn wij o...,Job description We are looking for an installe...,installer - glass interior walls,"{'glass', 'West Flanders region', 'GLASS INTER..."
11123,11123,https://www.vdab.be/vindeenjob/vacatures/68804505,heftruckchauffeur,forum jobs-food-construct roeselare,roeselare,Functieomschrijving Als heftruckchauffeur sta ...,Job description As a forklift driver you are r...,forklift driver,"{'equipment', '3rd grade', 'deviations Report ..."
11124,11124,https://www.vdab.be/vindeenjob/vacatures/68804506,technisch coördinator (+ bedrijfswagen),square city,evergem,Functieomschrijving Verwacht het onverwachte v...,Job description Expect the unexpected for your...,technical coordinator (+ company car),"{'Technical', 'maintenance', 'supervise', 'res..."
11125,11125,https://www.vdab.be/vindeenjob/vacatures/68804507,mechanieker,forum jobs-food-construct roeselare,roeselare,Functieomschrijving Als mechanieker behoort to...,"Job description As a mechanic, your duties inc...",mechanic,"{'equipment', 'mechanical pieces', 'maintenanc..."


In [23]:
df_jobs = df_jobs.merge(df_jobs2[['url','keywords']], on='url', )

In [18]:
df_jobs

,url,title,employer,location,description,description_translated,title_translated,province,category,keywords
0,https://www.vdab.be/vindeenjob/vacatures/68793122,technisch aankoopassistent,le grand & associates,kortrijk,Functieomschrijving LGA Engineering is een rek...,Job description LGA Engineering is a recruitme...,technical purchasing assistant,West Flanders,Administration,"{'Profile Profile Experience', 'recruitment', ..."
1,https://www.vdab.be/vindeenjob/vacatures/68793123,pricing analist,lga engineering,provincie vlaams-brabant,Functieomschrijving Heb jij een passie voor de...,Job description Do you have a passion for the ...,pricing analist,Flemish Brabant,"Finance, Insurance & law","{'Job', 'career', 'job', 'competencies', 'logi..."
2,https://www.vdab.be/vindeenjob/vacatures/68793124,pricing analist maritiem,lga engineering,antwerpen,Functieomschrijving Voor onze logistieke klant...,Job description For our logistics customer in ...,maritime pricing analyst,Antwerp,Education & training,"{'Job', 'Offer Offer', 'contract prices', 'Ana..."
3,https://www.vdab.be/vindeenjob/vacatures/68793125,warehouse teamleader,start people,herentals,Functieomschrijving Onze klant gevestigd in He...,Job description Our client based in Herentals ...,warehouse team leader,Antwerp,Education & training,"{'operating logistics', 'role', 'globally oper..."
4,https://www.vdab.be/vindeenjob/vacatures/68793126,grafisch vormgever,start people,pelt,Functieomschrijving 1 op de 10 gezinnen geniet...,Job description 1 in 10 families already enjoy...,graphical designer,Limburg,Administration,"{'Job', 'designing', 'wooden windows', 'doors'..."
...,...,...,...,...,...,...,...,...,...,...
11124,https://www.vdab.be/vindeenjob/vacatures/68804504,plaatser - glazen binnenwanden,forum construct maldegem,oostkamp,Functieomschrijving Voor onze klant zijn wij o...,Job description We are looking for an installe...,installer - glass interior walls,West Flanders,Construction & real estate,"{'glass', 'West Flanders region', 'GLASS INTER..."
11125,https://www.vdab.be/vindeenjob/vacatures/68804505,heftruckchauffeur,forum jobs-food-construct roeselare,roeselare,Functieomschrijving Als heftruckchauffeur sta ...,Job description As a forklift driver you are r...,forklift driver,West Flanders,Education & training,"{'equipment', '3rd grade', 'deviations Report ..."
11126,https://www.vdab.be/vindeenjob/vacatures/68804506,technisch coördinator (+ bedrijfswagen),square city,evergem,Functieomschrijving Verwacht het onverwachte v...,Job description Expect the unexpected for your...,technical coordinator (+ company car),East Flanders,Administration,"{'Technical', 'maintenance', 'supervise', 'res..."
11127,https://www.vdab.be/vindeenjob/vacatures/68804507,mechanieker,forum jobs-food-construct roeselare,roeselare,Functieomschrijving Als mechanieker behoort to...,"Job description As a mechanic, your duties inc...",mechanic,West Flanders,Executive board & management,"{'equipment', 'mechanical pieces', 'maintenanc..."


In [24]:
df_jobs.to_csv('/Users/ayushshrestha/Desktop/PPA/hr_tool_irakli/VDAB_data_translated_provinces_category_keywords.csv')